
**This notebook runs HBV over CAMELS-GB dataset catchments with less than 10% of missing data, and stock results (parameters and performance metrics) in HBV_Simulation_Data_CAMELS_GB.csv**

**Author:** Lionel Cedric Gohouede

## 1. MOUNT GOOGLE DRIVE

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. IMPORT LIBRARIES

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
import warnings
from numba import njit
warnings.filterwarnings('ignore')

print("✅ Libraries imported successfully")

✅ Libraries imported successfully


## 3. FILTER DATA BY TIME PERIOD (1990-2014)

In [ ]:
data_dir = '/content/drive/MyDrive/Colab Notebooks/CAMELS_GB/'
start_date, end_date = '1990-01-01', '2014-12-31'

print("Loading, filtering, and aligning CSV files...\n")

# 1. Fast Load & Filter
df_pcp  = pd.read_csv(f"{data_dir}pcp_mm.csv", index_col=0).loc[start_date:end_date]
df_pet  = pd.read_csv(f"{data_dir}pet_mm.csv", index_col=0).loc[start_date:end_date]
df_q    = pd.read_csv(f"{data_dir}q_cms_obs.csv", index_col=0).loc[start_date:end_date]  # m^3/s
df_area = pd.read_csv(f"{data_dir}topographic_attributes.csv", index_col=0)["area"]       # km²

# Ensure station IDs have the same type
df_pcp.columns  = df_pcp.columns.astype(str).str.strip()
df_pet.columns  = df_pet.columns.astype(str).str.strip()
df_q.columns    = df_q.columns.astype(str).str.strip()
df_area.index   = df_area.index.astype(str).str.strip()

# 2. Fast Alignment
common_stations = sorted(
    set(df_pcp.columns)
    & set(df_pet.columns)
    & set(df_q.columns)
    & set(df_area.index)
)

df_pcp  = df_pcp[common_stations]
df_pet  = df_pet[common_stations]
df_q    = df_q[common_stations]
df_area = df_area.loc[common_stations]

print(f"✅ Data aligned. Total common stations: {len(common_stations)}")

# ============================================
# Memory-Optimized Wrapper Classes
# ============================================
class SimpleArray:
    __slots__ = ["array"]

    def __init__(self, array):
        self.array = array

    def to_numpy(self):
        return self.array


class StationData:
    __slots__ = ["data"]

    def __init__(self, data_dict):
        self.data = data_dict

    def sel(self, dynamic_features=None):
        return SimpleArray(self.data[dynamic_features])

    def static(self, feature):
        return self.data[feature]


# ============================================
# Ultra-Fast Dictionary Construction
# ============================================
print("Building ds_recent dictionary...\n")

pcp_arr  = df_pcp.to_numpy()
pet_arr  = df_pet.to_numpy()
q_arr    = df_q.to_numpy()
area_arr = df_area.to_numpy()
date_arr = pd.to_datetime(df_pcp.index).to_numpy()

ds_recent = {
    st: StationData({
        "pcp_mm": pcp_arr[:, i],
        "pet_mm": pet_arr[:, i],
        "q_cms_obs": q_arr[:, i],
        "area_km2": area_arr[i],
        "date": date_arr,
    })
    for i, st in enumerate(common_stations)
}

print("✅ Dictionary built successfully!")

# ============================================
# Validation Test
# ============================================
test_station = common_stations[0]
print(f"\nTesting data access for station: {test_station}")

Q_obs = ds_recent[test_station].sel("q_cms_obs").to_numpy()
P     = ds_recent[test_station].sel("pcp_mm").to_numpy()
PET   = ds_recent[test_station].sel("pet_mm").to_numpy()
area  = ds_recent[test_station].static("area_km2")

print("✅ Extraction works correctly!")
print(f"   Q_obs shape: {Q_obs.shape}")
print(f"   Q_obs - min: {np.nanmin(Q_obs):.2f}, max: {np.nanmax(Q_obs):.2f}, mean: {np.nanmean(Q_obs):.2f}")
print(f"   Q_obs Missing: {np.sum(np.isnan(Q_obs))} ({np.sum(np.isnan(Q_obs))/len(Q_obs)*100:.1f}%)")
print(f"   Catchment area: {area:.2f} km²")

Loading, filtering, and aligning CSV files...

✅ Data aligned. Total common stations: 671
Building ds_recent dictionary...

✅ Dictionary built successfully!

Testing data access for station: 10002
✅ Extraction works correctly!
   Q_obs shape: (9131,)
   Q_obs - min: 0.80, max: 94.41, mean: 5.18
   Q_obs Missing: 0 (0.0%)
   Catchment area: 325.72 km²


## 4. MAIN CODE

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from numba import njit

# ============================================
# 1. COMPILED HBV MODEL (state simulation — inherently sequential)
# ============================================
@njit(cache=True)
def hbv_numba(P, ET, params, Qsim):
    """
    Runs the HBV bucket model in place into the preallocated `Qsim` buffer.
    Every index of Qsim is written unconditionally each call (real value or NaN),
    so buffer reuse across calls is numerically safe (no stale values persist).
    """
    FC, BETA, LP, K0, K1, K2, UZL, PERC = (
        params[0], params[1], params[2], params[3],
        params[4], params[5], params[6], params[7]
    )
    SM = 0.5 * FC
    UZ = 0.0
    LZ = 0.0
    n = len(P)

    for t in range(n):
        p = P[t]
        et = ET[t]

        if np.isnan(p) or np.isnan(et):
            Qsim[t] = np.nan
            continue

        soil_ratio = max(0.0, min(SM / FC, 1.0))
        recharge = p * soil_ratio ** BETA
        SM += p - recharge
        if SM > FC:
            recharge += SM - FC
            SM = FC
        SM = max(SM, 0.0)

        evap = min(et * min(SM / (LP * FC), 1.0), SM)
        SM -= evap
        SM = max(SM, 0.0)

        UZ += recharge
        perc = min(PERC, UZ)
        UZ -= perc
        LZ += perc

        Q0 = K0 * (UZ - UZL) if UZ > UZL else 0.0
        UZ -= Q0

        Q1 = K1 * UZ
        UZ -= Q1

        Q2 = K2 * LZ
        LZ -= Q2

        UZ = max(UZ, 0.0)
        LZ = max(LZ, 0.0)

        Qsim[t] = max(0.0, Q0 + Q1 + Q2)

    return Qsim  # mm/day


# ============================================
# 2. FAST NSE-ONLY OBJECTIVE (hot path — used inside scipy.optimize)
# ============================================
@njit(cache=True)
def objective_hbv_fast(params, P_train, ET_train, obs_train_filled_cms,
                       mask_train_f, denom_train_cms, Qsim_buffer,
                       area, conv):
    """
    Computes 1 - NSE on the training segment, with NSE evaluated in m^3/s.
    """
    Qsim_mmd = hbv_numba(P_train, ET_train, params, Qsim_buffer)
    Qsim_cms = Qsim_mmd * area / conv

    if denom_train_cms <= 0.0:
        return 1e9

    diff = (Qsim_cms - obs_train_filled_cms) * mask_train_f  # zero at invalid steps
    num = np.sum(diff ** 2)
    nse = 1.0 - num / denom_train_cms

    if not np.isfinite(nse):
        return 1e9
    return 1.0 - nse


# ============================================
# 3. FULL METRIC SUITE (post-hoc only — called twice per station, not in the hot loop)
# ============================================
def compute_pte(obs, sim, dates):
    """Computes Mean Absolute Annual Peak Timing Error (PTE) in days."""
    df = pd.DataFrame({'obs': obs, 'sim': sim}, index=pd.DatetimeIndex(dates))
    df = df.dropna()

    if df.empty:
        return np.nan

    pte_list = []
    for year, group in df.groupby(df.index.year):
        days_in_year = pd.Timestamp(year, 12, 31).dayofyear
        if len(group) < 0.8 * days_in_year:
            continue

        idx_obs_peak = group['obs'].idxmax()
        idx_sim_peak = group['sim'].idxmax()

        diff_days = abs((idx_sim_peak - idx_obs_peak).days)
        pte_list.append(diff_days)

    return np.mean(pte_list) if pte_list else np.nan


def compute_all_metrics(obs, sim, dates):
    """
    Computes NSE, NNSE, RMSE, PBIAS, FHV, FLV, KGE, and PTE exclusively on timesteps
    where BOTH obs and sim are valid.
    """
    mask = ~np.isnan(obs) & ~np.isnan(sim)
    o = obs[mask]
    s = sim[mask]
    L = o.size

    if L == 0:
        return dict(NSE=np.nan, NNSE=np.nan, RMSE=np.nan,
                    PBIAS=np.nan, FHV=np.nan, FLV=np.nan, KGE=np.nan, PTE=np.nan)

    obs_mean = np.mean(o)
    sim_mean = np.mean(s)
    denom = np.sum((o - obs_mean) ** 2)

    # --- NSE / NNSE ---
    if denom == 0.0:
        nse = np.nan
        nnse = np.nan
    else:
        nse = 1.0 - np.sum((s - o) ** 2) / denom
        nnse = 1.0 / (2.0 - nse) if np.isfinite(nse) else np.nan

    # --- RMSE ---
    rmse = np.sqrt(np.mean((s - o) ** 2))

    # --- PBIAS (Yilmaz 2008) ---
    sum_obs = np.sum(o)
    pbias = 100.0 * np.sum(s - o) / sum_obs if sum_obs != 0.0 else np.nan

    # --- FHV (top 2%) / FLV (bottom 30%) (Yilmaz 2008) ---
    o_sorted = np.sort(o)[::-1]
    s_sorted = np.sort(s)[::-1]

    n_hv = max(1, int(round(0.02 * L)))
    sum_obs_hv = np.sum(o_sorted[:n_hv])
    sum_sim_hv = np.sum(s_sorted[:n_hv])
    fhv = 100.0 * (sum_sim_hv - sum_obs_hv) / sum_obs_hv if sum_obs_hv != 0.0 else np.nan

    n_lv = max(1, int(round(0.30 * L)))
    epsilon = 1e-6
    o_low = o_sorted[-n_lv:]
    s_low = s_sorted[-n_lv:]

    log_o = np.log(o_low + epsilon)
    log_s = np.log(s_low + epsilon)
    log_o_L = np.log(o_sorted[-1] + epsilon)
    log_s_L = np.log(s_sorted[-1] + epsilon)

    term_o = np.sum(log_o - log_o_L)
    term_s = np.sum(log_s - log_s_L)
    flv = -100.0 * (term_s - term_o) / term_o if term_o != 0.0 else np.nan

    # --- KGE (Gupta et al., 2009) ---
    sigma_obs = np.std(o)
    sigma_sim = np.std(s)
    if sigma_obs == 0.0 or sigma_sim == 0.0 or obs_mean == 0.0:
        kge = np.nan
    else:
        r = np.corrcoef(o, s)[0, 1]
        alpha = sigma_sim / sigma_obs
        beta = sim_mean / obs_mean
        if not np.isfinite(r):
            kge = np.nan
        else:
            kge = 1.0 - np.sqrt((r - 1.0) ** 2 + (alpha - 1.0) ** 2 + (beta - 1.0) ** 2)

    # --- Peak Time Error (Annual) ---
    pte = compute_pte(obs, sim, dates)

    return dict(NSE=nse, NNSE=nnse, RMSE=rmse, PBIAS=pbias,
                FHV=fhv, FLV=flv, KGE=kge, PTE=pte)

# ============================================
# 4. EXECUTION LOOP
# ============================================
all_stations = list(ds_recent.keys())
b1_ratio = 0.7
max_missing_ratio = 0.1
results = {}

param_names  = ["FC", "BETA", "LP", "K0", "K1", "K2", "UZL", "PERC"]

param_bounds = [
    (50, 1000), (1, 6), (0.3, 1), (0, 0.5),
    (0, 0.3), (0, 0.1), (0, 100), (0, 5)
]

CONV = 86.4  # mm/day * km^2 / CONV = m^3/s

n_skip_no_area    = 0
n_skip_missing    = 0
n_skip_no_data    = 0
n_skip_calib_fail = 0

for i, station_id in enumerate(all_stations, 1):
    print(f"\n=== Station {station_id} ===, Number={i}")

    Q_obs_cms = ds_recent[station_id].sel(dynamic_features="q_cms_obs").to_numpy()  # m³/s
    P    = ds_recent[station_id].sel(dynamic_features="pcp_mm").to_numpy()          # mm/day
    ET   = ds_recent[station_id].sel(dynamic_features="pet_mm").to_numpy()          # mm/day
    area = ds_recent[station_id].static("area_km2")                                 # km²

    dates = ds_recent[station_id].sel(dynamic_features="date").to_numpy()
    time_months = pd.DatetimeIndex(dates).month.to_numpy()

    if area is None or not np.isfinite(area) or area <= 0.0:
        print("⚠️  Skipped — missing or invalid catchment area.")
        n_skip_no_area += 1
        continue

    N = len(Q_obs_cms)

    if N == 0 or np.all(np.isnan(Q_obs_cms)):
        print("⚠️ Station skipped (no valid data).")
        n_skip_no_data += 1
        continue

    missing_count = np.sum(np.isnan(Q_obs_cms))
    missing_ratio = missing_count / N
    if missing_ratio > max_missing_ratio:
        print(f"⚠️ Too many missing values ({missing_ratio*100:.1f}%)")
        n_skip_missing += 1
        continue

    # Global Runoff Ratio calculation (mm to mm representation)
    total_p = np.nansum(P)
    total_q_mm = np.nansum(Q_obs_cms * CONV / area)
    RR = total_q_mm / total_p if total_p > 0 else np.nan

    b1 = int(N * b1_ratio)

    P_train  = P[:b1]
    ET_train = ET[:b1]
    Q_obs_train_cms = Q_obs_cms[:b1]   # calibration domain is now m³/s

    # --- Precompute everything that does NOT depend on the optimization parameters ---
    mask_train = (~np.isnan(Q_obs_train_cms)
                  & ~np.isnan(P_train)
                  & ~np.isnan(ET_train))
    mask_train_f = mask_train.astype(np.float64)
    obs_train_filled_cms = np.where(mask_train, Q_obs_train_cms, 0.0)

    valid_obs_train = Q_obs_train_cms[mask_train]
    if valid_obs_train.size == 0:
        print("⚠️ No valid training observations after masking.")
        n_skip_no_data += 1
        continue
    obs_mean_train = np.mean(valid_obs_train)
    denom_train_cms = np.sum((valid_obs_train - obs_mean_train) ** 2)

    Qsim_train_buffer = np.empty(b1, dtype=np.float64)

    # Multi-start calibration (NSE evaluated in m^3/s)
    best_fun = np.inf
    best_x = None
    np.random.seed(42)
    for _ in range(10):
        x0 = np.array([np.random.uniform(b[0], b[1]) for b in param_bounds])
        res = minimize(
            objective_hbv_fast,
            x0,
            args=(P_train, ET_train, obs_train_filled_cms, mask_train_f,
                  denom_train_cms, Qsim_train_buffer, area, CONV),
            method="L-BFGS-B",
            bounds=param_bounds,
            options={"maxiter": 3000}
        )
        if res.fun < best_fun:
            best_fun = res.fun
            best_x = res.x

    if best_x is None:
        print("⚠️ Calibration failed.")
        n_skip_calib_fail += 1
        continue

    # Full-period simulation, mm/day
    Qsim_full_buffer = np.empty(N, dtype=np.float64)
    Qsim_mmd = hbv_numba(P, ET, best_x, Qsim_full_buffer).copy()

    # Convert sim mm/day → m³/s for metrics; obs already in m³/s
    Qsim_cms = Qsim_mmd * area / CONV

    m_train = compute_all_metrics(Q_obs_train_cms, Qsim_cms[:b1], dates[:b1])
    m_val   = compute_all_metrics(Q_obs_cms[b1:], Qsim_cms[b1:], dates[b1:])

    print(f"✅ Training NSE: {m_train['NSE']:.3f}, Validation NSE: {m_val['NSE']:.3f}")
    print(f"   Training KGE: {m_train['KGE']:.3f}, Validation KGE: {m_val['KGE']:.3f}")
    print(f"   Training RMSE: {m_train['RMSE']:.3f} m³/s, Validation RMSE: {m_val['RMSE']:.3f} m³/s")
    print(f"   Training PBIAS: {m_train['PBIAS']:.3f}, Validation PBIAS: {m_val['PBIAS']:.3f}")
    print(f"   Params: FC={best_x[0]:.2f}, BETA={best_x[1]:.2f}, LP={best_x[2]:.2f}, "
          f"K0={best_x[3]:.3f}, K1={best_x[4]:.3f}, K2={best_x[5]:.3f}, "
          f"UZL={best_x[6]:.2f}, PERC={best_x[7]:.2f}")

    station_result = {
        "params": best_x.tolist(),
        "param_names":   param_names,
        "NSE_train": m_train["NSE"],   "NSE_val": m_val["NSE"],
        "NNSE_train": m_train["NNSE"], "NNSE_val": m_val["NNSE"],
        "RMSE_train": m_train["RMSE"], "RMSE_val": m_val["RMSE"],
        "PBIAS_train": m_train["PBIAS"], "PBIAS_val": m_val["PBIAS"],
        "FHV_train": m_train["FHV"],   "FHV_val": m_val["FHV"],
        "FLV_train": m_train["FLV"],   "FLV_val": m_val["FLV"],
        "KGE_train": m_train["KGE"],   "KGE_val": m_val["KGE"],
        "PTE_train": m_train["PTE"],   "PTE_val": m_val["PTE"],
        "RR": RR,
        "Qsim_mmd": Qsim_mmd,
        "Qsim_cms": Qsim_cms,
        "Q_obs_cms": Q_obs_cms,
        "missing_ratio": missing_ratio,
        "missing_count": missing_count,
    }

    # -----------------------------------------------------------------
    # Extract Seasonal Metrics
    # -----------------------------------------------------------------
    seasons_map = {
        "DJF": [12, 1, 2],
        "MAM": [3, 4, 5],
        "JJA": [6, 7, 8],
        "SON": [9, 10, 11]
    }

    months_train = time_months[:b1]
    months_val = time_months[b1:]

    for season, m_list in seasons_map.items():
        mask_train_season = np.isin(months_train, m_list)
        mask_val_season = np.isin(months_val, m_list)

        if np.any(mask_train_season):
            s_train_metrics = compute_all_metrics(Q_obs_train_cms[mask_train_season], Qsim_cms[:b1][mask_train_season], dates[:b1][mask_train_season])
        else:
            s_train_metrics = {k: np.nan for k in m_train.keys()}

        if np.any(mask_val_season):
            s_val_metrics = compute_all_metrics(Q_obs_cms[b1:][mask_val_season], Qsim_cms[b1:][mask_val_season], dates[b1:][mask_val_season])
        else:
            s_val_metrics = {k: np.nan for k in m_val.keys()}

        station_result[f"NSE_{season}_train"] = s_train_metrics["NSE"]
        station_result[f"NSE_{season}_val"] = s_val_metrics["NSE"]
        station_result[f"KGE_{season}_train"] = s_train_metrics["KGE"]
        station_result[f"KGE_{season}_val"] = s_val_metrics["KGE"]
        station_result[f"RMSE_{season}_train"] = s_train_metrics["RMSE"]
        station_result[f"RMSE_{season}_val"] = s_val_metrics["RMSE"]
        station_result[f"PBIAS_{season}_train"] = s_train_metrics["PBIAS"]
        station_result[f"PBIAS_{season}_val"] = s_val_metrics["PBIAS"]

    results[station_id] = station_result

n_total   = len(all_stations)
n_success = len(results)
print(f"\n✅ Simulation done for {n_success}/{n_total} valid catchments (≤10% de NaN).")
print(f"   Skipped — no/invalid area  : {n_skip_no_area}")
print(f"   Skipped — too many NaN     : {n_skip_missing}")
print(f"   Skipped — no valid obs     : {n_skip_no_data}")
print(f"   Skipped — calibration fail : {n_skip_calib_fail}")


=== Station 10002 ===, Number=1
✅ Training NSE: 0.717, Validation NSE: 0.580
   Training KGE: 0.762, Validation KGE: 0.635
   Training RMSE: 2.776 m³/s, Validation RMSE: 3.991 m³/s
   Training PBIAS: 1.422, Validation PBIAS: 2.942
   Params: FC=166.85, BETA=1.61, LP=0.82, K0=0.182, K1=0.142, K2=0.027, UZL=12.55, PERC=1.48

=== Station 10003 ===, Number=2
✅ Training NSE: 0.849, Validation NSE: 0.772
   Training KGE: 0.874, Validation KGE: 0.804
   Training RMSE: 2.757 m³/s, Validation RMSE: 3.676 m³/s
   Training PBIAS: -2.201, Validation PBIAS: 6.143
   Params: FC=261.05, BETA=3.94, LP=1.00, K0=0.127, K1=0.105, K2=0.036, UZL=17.63, PERC=2.34

=== Station 1001 ===, Number=3
⚠️ Too many missing values (23.4%)

=== Station 101002 ===, Number=4
✅ Training NSE: 0.757, Validation NSE: 0.725
   Training KGE: 0.859, Validation KGE: 0.862
   Training RMSE: 0.187 m³/s, Validation RMSE: 0.209 m³/s
   Training PBIAS: -2.212, Validation PBIAS: -0.644
   Params: FC=627.74, BETA=1.00, LP=0.35, K0=0.

## 5. SAVE SUMMARY

In [ ]:
import os
import pandas as pd
import numpy as np
from google.colab import drive

# ============================================
# 1. EXTRACT DATA TO DATAFRAME
# ============================================
rows = []
seasons = ["DJF", "MAM", "JJA", "SON"]

for station_id, res in results.items():
    # Dynamically unpack parameters using param_names
    param_dict = dict(zip(res["param_names"], res["params"]))

    row = {
        "station_id": station_id,
        **param_dict,
        "RR": res.get("RR", np.nan),
        "NSE_train": res.get("NSE_train", np.nan),
        "NSE_val": res.get("NSE_val", np.nan),
        "NNSE_train": res.get("NNSE_train", np.nan),
        "NNSE_val": res.get("NNSE_val", np.nan),
        "RMSE_train": res.get("RMSE_train", np.nan),
        "RMSE_val": res.get("RMSE_val", np.nan),
        "PBIAS_train": res.get("PBIAS_train", np.nan),
        "PBIAS_val": res.get("PBIAS_val", np.nan),
        "FHV_train": res.get("FHV_train", np.nan),
        "FHV_val": res.get("FHV_val", np.nan),
        "FLV_train": res.get("FLV_train", np.nan),
        "FLV_val": res.get("FLV_val", np.nan),
        "KGE_train": res.get("KGE_train", np.nan),
        "KGE_val": res.get("KGE_val", np.nan),
        "PTE_train": res.get("PTE_train", np.nan),
        "PTE_val": res.get("PTE_val", np.nan),
        "missing_ratio": res.get("missing_ratio", np.nan),
        "missing_count": res.get("missing_count", np.nan),
    }

    # Dynamically add seasonal metrics
    for season in seasons:
        row[f"NSE_{season}_train"]   = res.get(f"NSE_{season}_train", np.nan)
        row[f"NSE_{season}_val"]     = res.get(f"NSE_{season}_val", np.nan)
        row[f"KGE_{season}_train"]   = res.get(f"KGE_{season}_train", np.nan)
        row[f"KGE_{season}_val"]     = res.get(f"KGE_{season}_val", np.nan)
        row[f"RMSE_{season}_train"]  = res.get(f"RMSE_{season}_train", np.nan)
        row[f"RMSE_{season}_val"]    = res.get(f"RMSE_{season}_val", np.nan)
        row[f"PBIAS_{season}_train"] = res.get(f"PBIAS_{season}_train", np.nan)
        row[f"PBIAS_{season}_val"]   = res.get(f"PBIAS_{season}_val", np.nan)

    rows.append(row)

df = pd.DataFrame(rows)

# ============================================
# 2. SAVE RESULTS
# ============================================
# Local save
df.to_csv("HBV_Simulation_Data_CAMELS_GB.csv", index=False)
print("✅ Saved locally to HBV_Simulation_Data_CAMELS_GB.csv")

# Save to Google Drive (this is the file the loader script reads back)
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount("/content/drive")

df.to_csv("/content/drive/MyDrive/Colab Notebooks/Data/HBV_Simulation_Data_CAMELS_GB.csv", index=False)
print("✅ Saved to Google Drive")

# ============================================
# 3. DEFINE METRICS TO REPORT
# ============================================
global_metrics = [
    ("NSE_train",   "NSE Training",     ""),
    ("NSE_val",     "NSE Validation",   ""),
    ("NNSE_train",  "NNSE Training",    ""),
    ("NNSE_val",    "NNSE Validation",  ""),
    ("KGE_train",   "KGE Training",     ""),
    ("KGE_val",     "KGE Validation",   ""),
    ("RMSE_train",  "RMSE Training",    " m3/s"),
    ("RMSE_val",    "RMSE Validation",  " m3/s"),
    ("PBIAS_train", "PBIAS Training",   "%"),
    ("PBIAS_val",   "PBIAS Validation", "%"),
    ("FHV_train",   "FHV Training",     "%"),
    ("FHV_val",     "FHV Validation",   "%"),
    ("FLV_train",   "FLV Training",     "%"),
    ("FLV_val",     "FLV Validation",   "%"),
    ("PTE_train",   "PTE Training",     " days"),
    ("PTE_val",     "PTE Validation",   " days"),
]

season_vars = [("NSE", ""), ("KGE", ""), ("RMSE", " m3/s"), ("PBIAS", "%")]
seasonal_metrics = []
for var, unit in season_vars:
    for s in seasons:
        for split, label in [("train", "Training"), ("val", "Validation")]:
            seasonal_metrics.append((f"{var}_{s}_{split}", f"{var} {s} {label}", unit))

# ============================================
# 4. PRINT STATISTICS
# ============================================
def print_stats(metrics_list, df, title):
    print(f"\n{'='*60}")
    print(f" {title}")
    print(f"{'='*60}")
    for key, label, unit in metrics_list:
        if key not in df.columns:
            print(f"  ⚠️  '{key}' not found in CSV — skipping.")
            continue
        values = pd.to_numeric(df[key], errors="coerce").dropna().to_numpy()
        n_valid   = len(values)
        n_dropped = len(df) - n_valid
        if n_valid == 0:
            print(f"  ⚠️  {label}: all NaN — no valid stations.")
            continue
        print(
            f"  {label:<28} | "
            f"Mean: {values.mean():8.3f}{unit}  "
            f"Median: {np.median(values):8.3f}{unit}  "
            f"Min: {values.min():8.3f}{unit}  "
            f"Max: {values.max():8.3f}{unit}  "
            f"P05: {np.percentile(values,  5):8.3f}{unit}  "
            f"P95: {np.percentile(values, 95):8.3f}{unit}  "
            f"(n={n_valid})"
        )
        if n_dropped:
            print(f"    ⚠️  {n_dropped} station(s) excluded (NaN).")

print(f"\n✅ Saved {len(df)} stations to CSV.")
print_stats(global_metrics,   df, "GLOBAL SUMMARY STATISTICS")
print_stats(seasonal_metrics, df, "SEASONAL SUMMARY STATISTICS")

✅ Saved locally to HBV_Simulation_Data_CAMELS_GB.csv
✅ Saved to Google Drive

✅ Saved 623 stations to CSV.

 GLOBAL SUMMARY STATISTICS
  NSE Training                 | Mean:    0.772  Median:    0.800  Min:   -3.449  Max:    0.933  P05:    0.602  P95:    0.900  (n=623)
  NSE Validation               | Mean:    0.713  Median:    0.759  Min:   -2.278  Max:    0.939  P05:    0.419  P95:    0.887  (n=623)
  NNSE Training                | Mean:    0.824  Median:    0.833  Min:    0.184  Max:    0.937  P05:    0.715  P95:    0.909  (n=623)
  NNSE Validation              | Mean:    0.792  Median:    0.806  Min:    0.234  Max:    0.942  P05:    0.633  P95:    0.899  (n=623)
  KGE Training                 | Mean:    0.829  Median:    0.845  Min:   -0.230  Max:    0.959  P05:    0.695  P95:    0.933  (n=623)
  KGE Validation               | Mean:    0.753  Median:    0.784  Min:   -0.679  Max:    0.948  P05:    0.506  P95:    0.900  (n=623)
  RMSE Training                | Mean:    3.987 m3/s  M